<a href="https://colab.research.google.com/github/hbisgin/DeepLearning/blob/main/DL_20_NormalizationSkippingResidualNetwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/EdwardRaff/Inside-Deep-Learning/refs/heads/main/idlmam.py

In [ ]:
import sys
sys.path.append('/content/idlmam.py')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader

from tqdm.autonotebook import tqdm

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

import pandas as pd

from sklearn.metrics import accuracy_score

import time

from idlmam import train_network, Flatten, weight_reset, set_seed
from idlmam import LanguageNameDataset, pad_and_pack, EmbeddingPackable, LastTimeStep, LambdaLayer

In [ ]:
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('png', 'pdf')

In [ ]:
torch.backends.cudnn.deterministic=True
set_seed(42)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
train_data = torchvision.datasets.FashionMNIST("./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.FashionMNIST("./", train=True, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)

In [ ]:
#Whats the width and height of our images?
W, H = 28, 28 #
#How many values are in the input? We use this to help determine the size of subsequent layers
D = 28*28 #28 * 28 images
#Hidden layer size
n = 256
#How many channels are in the input?
C = 1
#how many filters per convolutional layer
n_filters = 32
#How many classes are there?
classes = 10#

Let's do the baselines first

In [ ]:
fc_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(D,  n), nn.Tanh(), #First hidden layer
    *[nn.Sequential(nn.Linear(n,  n),nn.Tanh()) for _ in range(5)], #Now that each remaining layer has the same input/output sizes, we can make them with a list unpacking
    nn.Linear(n, classes),
)

In [ ]:
cnn_model = nn.Sequential(
    nn.Conv2d(C, n_filters, 3, padding=1),             nn.Tanh(),
    nn.Conv2d(n_filters, n_filters, 3, padding=1),     nn.Tanh(),
    nn.Conv2d(n_filters, n_filters, 3, padding=1),     nn.Tanh(),
    nn.MaxPool2d((2,2)),
    nn.Conv2d(  n_filters, 2*n_filters, 3, padding=1), nn.Tanh(),
    nn.Conv2d(2*n_filters, 2*n_filters, 3, padding=1), nn.Tanh(),
    nn.Conv2d(2*n_filters, 2*n_filters, 3, padding=1), nn.Tanh(),
    nn.MaxPool2d((2,2)),
    nn.Conv2d(2*n_filters, 4*n_filters, 3, padding=1), nn.Tanh(),
    nn.Conv2d(4*n_filters, 4*n_filters, 3, padding=1), nn.Tanh(),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

In [ ]:
loss_func = nn.CrossEntropyLoss()
fc_results = train_network(fc_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
cnn_results = train_network(cnn_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)

In [ ]:
del fc_model
del cnn_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=fc_results, label='Fully Connected')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_results, label='CNN')

In [ ]:
leak_rate=0.1

Before going further, let's do the leakyReLU normalized again




In [ ]:
fc_relu_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(D,  n), nn.LeakyReLU(leak_rate),
    *[nn.Sequential(nn.Linear(n,  n), nn.LeakyReLU(leak_rate)) for _ in range(5)],
    nn.Linear(n, classes),
)

In [ ]:
def cnnLayer(in_filters, out_filters=None, kernel_size=3):
    """
    in_filters: how many channels are coming into the layer
    out_filters: how many channels this layer should learn / output, or `None` if we want to have the same number of channels as the input.
    kernel_size: how large the kernel should be
    """
    if out_filters is None:
        out_filters = in_filters #This is a common pattern, so lets automate it as a default if not asked
    padding=kernel_size//2 #padding to stay the same size
    return nn.Sequential( # Combine the layer and activation into a single unit
        nn.Conv2d(in_filters, out_filters, kernel_size, padding=padding),
        nn.LeakyReLU(leak_rate)
    )

In [ ]:
cnn_relu_model = nn.Sequential(
    cnnLayer(C, n_filters), cnnLayer(n_filters), cnnLayer(n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(n_filters, 2*n_filters), cnnLayer(2*n_filters), cnnLayer(2*n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(2*n_filters, 4*n_filters), cnnLayer(4*n_filters),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

In [ ]:
fc_relu_results = train_network(fc_relu_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del fc_relu_model
cnn_relu_results = train_network(cnn_relu_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del cnn_relu_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=fc_results, label='FC')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_relu_results, label='FC-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_results, label='CNN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')

With batch normalization

In [ ]:
fc_bn_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(D,  n), nn.BatchNorm1d(n), nn.LeakyReLU(leak_rate),
    *[nn.Sequential(nn.Linear(n,  n), nn.BatchNorm1d(n), nn.LeakyReLU(leak_rate)) for _ in range(5)],
    nn.Linear(n, classes),
)

In [ ]:
def cnnLayer(in_filters, out_filters=None, kernel_size=3):
    if out_filters is None:
        out_filters = in_filters #This is a common pattern, so lets automate it as a default if not asked
    padding=kernel_size//2 #padding to stay the same size
    return nn.Sequential( # Combine the layer and activation into a single unit
        nn.Conv2d(in_filters, out_filters, kernel_size, padding=padding),
        nn.BatchNorm2d(out_filters), #The only change, adding BatchNorm2d after our convolution!
        nn.LeakyReLU(leak_rate)
    )

In [ ]:
cnn_bn_model = nn.Sequential(
    cnnLayer(C, n_filters), cnnLayer(n_filters), cnnLayer(n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(n_filters, 2*n_filters), cnnLayer(2*n_filters), cnnLayer(2*n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(2*n_filters, 4*n_filters), cnnLayer(4*n_filters),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

In [ ]:
fc_bn_results = train_network(fc_bn_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del fc_bn_model
cnn_bn_results = train_network(cnn_bn_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del cnn_bn_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=fc_relu_results, label='FC-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_bn_results, label='FC-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_bn_results, label='CNN-ReLU-BN')

#Now with LayerNorm

In [ ]:
fc_ln_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(D,  n), nn.LayerNorm([n]), nn.LeakyReLU(leak_rate),
    *[nn.Sequential(nn.Linear(n,  n), nn.LayerNorm([n]), nn.LeakyReLU(leak_rate)) for _ in range(5)],
    nn.Linear(n, classes),
)

In [ ]:
def cnnLayer(in_filters, out_filters=None, pool_factor=0,kernel_size=3):
    if out_filters is None:
        out_filters = in_filters #This is a common pattern, so lets automate it as a default if not asked
    padding=kernel_size//2 #padding to stay the same size
    return nn.Sequential( # Combine the layer and activation into a single unit
        nn.Conv2d(in_filters, out_filters, kernel_size, padding=padding),
        nn.LayerNorm([out_filters, W//(2**pool_factor), H//(2**pool_factor)]), #The only change, adding BatchNorm2d after our convolution!
        nn.LeakyReLU(leak_rate)
    )

In [ ]:
cnn_ln_model = nn.Sequential(
    cnnLayer(C, n_filters),
    cnnLayer(n_filters),
    cnnLayer(n_filters),
    nn.MaxPool2d((2,2)), #we've done one round of pooling, so , pool_factor=1 now
    cnnLayer(n_filters, 2*n_filters, pool_factor=1),
    cnnLayer(2*n_filters, pool_factor=1),
    cnnLayer(2*n_filters, pool_factor=1),
    nn.MaxPool2d((2,2)), #now we've done two rounds of pooling, so pool_factor=2
    cnnLayer(2*n_filters, 4*n_filters, pool_factor=2),
    cnnLayer(4*n_filters, pool_factor=2),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

In [ ]:
fc_ln_results = train_network(fc_ln_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del fc_ln_model
cnn_ln_results = train_network(cnn_ln_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del cnn_ln_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=fc_relu_results, label='FC-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_bn_results, label='FC-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_bn_results, label='CNN-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_ln_results, label='FC-ReLU-LN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_ln_results, label='CNN-ReLU-LN')

#Skipping Skipping

A PyTorch module defining a class for creating skip connections.


*   It will create one larger block of multiple layers in a “dense” style skip connection, containing `n_layers` total.
*   Used on its own it can create dense networks, or used sequentially it can create staggered skip connections.




In [ ]:
class SkipFC(nn.Module):
    def __init__(self, n_layers, in_size, out_size, leak_rate=0.1):
        """
        n_layers: how many hidden layers for this block of dense skip connections
        in_size: how many features are coming into this layer
        out_size: how many features should be used for the final layer of this block.
        leak_rate: the parameter for the LeakyReLU activation function.
        """
        super().__init__()

        #The final layer will be treated differently, so lets grab it's index to use in the next two lines
        l = n_layers-1
        #The linear and batch norm layers are stored in `layers` and `bns` respectively.
        #A list comprehensive creates all the layers in one line each.
        #The `if i == l` allows us to single out the last layer which needs to use `out_size` instead of `in_size`

        #NOTE THE MODULELIST here for both Linear and BN Layers

        #self.layers = nn.ModuleList([nn.Linear(in_size*l, out_size) if i == l else nn.Linear(in_size, in_size) for i in range(n_layers)])
        #self.bns = nn.ModuleList([nn.BatchNorm1d(out_size) if i == l else nn.BatchNorm1d(in_size) for i in range(n_layers)])

        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()

        l = n_layers - 1

        for i in range(n_layers):
            if i == l:
                self.layers.append(nn.Linear(in_size * l, out_size))
                self.bns.append(nn.BatchNorm1d(out_size))
            else:
                self.layers.append(nn.Linear(in_size, in_size))
                self.bns.append(nn.BatchNorm1d(in_size))

                #Since we are writing our own `forward` function instead of using nn.Sequential, we can just use one activation object multiple times.
        self.activation = nn.LeakyReLU(leak_rate)

    def forward(self, x):
        #First we need a location to store the activations from every layer (except the last one) in this block.
        #All the activations will be combined as the input to the last layer, which is what makes the skips!

        activations = []

        #zip the linear and normalization layers into paired tuples, using [:-1] to select all but the last item in each list.

        for layer, bn in zip(self.layers[:-1], self.bns[:-1]):
            x = self.activation(bn(layer(x))) #We are chaining layer->batchnorm->activation
            activations.append( x ) #intermediate outputs are stored in activations list.



        x = torch.cat(activations, dim=1) #concatenate the activations together, this makes the input for the last layer

        #Now manually use the last linear and batch-norm layer on this concatenated input, giving us the result.

        return self.activation(self.bns[-1](self.layers[-1](x))) #this part is done separately because
        #takes concatenated activations
        #has different input size
        #produces the block output

In [ ]:
loss_func = nn.CrossEntropyLoss()

In [ ]:
fc_skip_model = nn.Sequential(
    nn.Flatten(),
    SkipFC(2, D, n),
    SkipFC(2, n, n),
    SkipFC(2, n, n),
    nn.Linear(n, classes),
)

fc_skip_results = train_network(fc_skip_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del fc_skip_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=fc_relu_results, label='FC-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_bn_results, label='FC-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=fc_skip_results, label='FC-ReLU-BN-Skip')

Skipping w/ CNN

In [ ]:
class SkipConv2d(nn.Module):
    def __init__(self, n_layers, in_channels, out_channels, kernel_size=3, leak_rate=0.1):
        super().__init__()

        #The last convolution will have a different number of inputs and output channels, so we still need that index
        l = n_layers-1
        #this is just simple helper values
        f = (kernel_size, kernel_size)
        pad = (kernel_size-1)//2

        #Defining the layers used, altering the construction of the last layer using the same `if i == l` list comprehension. We are going to combine convolutions via their channels, so the in and out channels change for the last layer.
        self.layers = nn.ModuleList([nn.Conv2d(in_channels*l, out_channels, kernel_size=f, padding=pad) if i == l else nn.Conv2d(in_channels, in_channels, kernel_size=f, padding=pad) for i in range(n_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm2d(out_channels) if i == l else nn.BatchNorm2d(in_channels) for i in range(n_layers)])

        self.activation = nn.LeakyReLU(leak_rate)

    def forward(self, x):
        #This code is identical to the SkipFC class, but its worth highliting the most important line that could change.
        activations = []

        for layer, bn in zip(self.layers[:-1], self.bns[:-1]):
            x = self.activation(bn(layer(x)))
            activations.append( x )
        #Which is the concatenation of all the activations here. Our tensors are organized as (B, C, W, H), which is the default in PyTorch. But you can change that, and sometimes people use (B, W, H, C). In that situation the C channel is at index 3 instead of 1. So you would change `cat=3` in that scenario. This is also how you would adapt this code to work with RNNs
        x = torch.cat(activations, dim=1)

        return self.activation(self.bns[-1](self.layers[-1](x)))

In [ ]:
cnn_skip_model = nn.Sequential(
    nn.Conv2d(C, n_filters, (3,3), padding=1),
    SkipConv2d(3, n_filters, 2*n_filters),
    nn.MaxPool2d((2,2)),
    nn.LeakyReLU(),
    SkipConv2d(3, 2*n_filters, 4*n_filters),
    nn.MaxPool2d((2,2)),
    SkipConv2d(2, 4*n_filters, 4*n_filters),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

cnn_skip_results = train_network(cnn_skip_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del cnn_skip_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_bn_results, label='CNN-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_skip_results, label='CNN-ReLU-BN-Skip')

#1 × 1 Convolutions: Sharing and reshaping information in channels

In [ ]:
def infoShareBlock(n_filters):
    return nn.Sequential(
        nn.Conv2d(n_filters, n_filters, (1,1), padding=0),
        nn.BatchNorm2d(n_filters),
        nn.LeakyReLU())

In [ ]:
def cnnLayer(in_filters, out_filters=None, kernel_size=3):
    if out_filters is None:
        out_filters = in_filters #This is a common pattern, so lets automate it as a default if not asked
    padding=kernel_size//2 #padding to stay the same size
    return nn.Sequential( # Combine the layer and activation into a single unit
        nn.Conv2d(in_filters, out_filters, kernel_size, padding=padding),
        nn.BatchNorm2d(out_filters), #The only change, adding BatchNorm2d after our convolution!
        nn.LeakyReLU(leak_rate)
    )

In [ ]:
cnn_1x1_model = nn.Sequential(
    cnnLayer(C, n_filters),
    cnnLayer(n_filters),
    infoShareBlock(n_filters), #first info block after 2x cnnLayers
    cnnLayer(n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(n_filters, 2*n_filters),
    cnnLayer(2*n_filters),
    infoShareBlock(2*n_filters),
    cnnLayer(2*n_filters),
    nn.MaxPool2d((2,2)),
    cnnLayer(2*n_filters, 4*n_filters),
    cnnLayer(4*n_filters),
    infoShareBlock(4*n_filters),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)
#Now train up this model
cnn_1x1_results = train_network(cnn_1x1_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)
del cnn_1x1_model

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_bn_results, label='CNN-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_1x1_results, label='CNN-ReLU-BN-1x1')

Residual Block: Type E, one of the favored residual setups

In [ ]:
class ResidualBlockE(nn.Module):
    def __init__(self, channels, kernel_size=3, leak_rate=0.1):
        """
        channels: how many channels are in the input/output to this layer
        kernel_size: how large of a filter should we use
        leak_rate: paramter for the LeakyReLU activation function
        """
        super().__init__()
        #how much padding will our convolutional layers need to maintain the input shape
        pad = (kernel_size-1)//2

        #Define the conv an BN layers we will use in a sub-network, just 2 hidden layers of conv/BN/activation
        self.F = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(leak_rate),
            nn.Conv2d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(leak_rate),
        )

    def forward(self, x):
        return x + self.F(x) #F() has all the work for the long path, we just add it to the input

Residual BottleNect

In [ ]:
class ResidualBottleNeck(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, leak_rate=0.1):
        super().__init__()
        #how much padding will our convolutional layers need to maintain the input shape
        pad = (kernel_size-1)//2
        #The botteneck should be smaller, so output/4 or input. You could also try changing max to min, its not a major issue.
        bottleneck = max(out_channels//4, in_channels)
        #Define the three sets of BN and convolution layers we need.
        #Notice that for the 1x1 convs we use padding=0, because 1x1 will not change shape!
        self.F = nn.Sequential(
            #Compress down
            nn.BatchNorm2d(in_channels),
            nn.LeakyReLU(leak_rate),
            nn.Conv2d(in_channels, bottleneck, 1, padding=0),
            #Normal layer doing a full conv
            nn.BatchNorm2d(bottleneck),
            nn.LeakyReLU(leak_rate),
            nn.Conv2d(bottleneck, bottleneck, kernel_size, padding=pad),
            #Expand back up
            nn.BatchNorm2d(bottleneck),
            nn.LeakyReLU(leak_rate),
            nn.Conv2d(bottleneck, out_channels, 1, padding=0)
        )

        #By default, our shortcut will be the identiy function - which simply returns the input as the output
        self.shortcut = nn.Identity()
        #If we need to change the shape, then lets turn the shortcut into a small layer with 1x1 conv and BM
        if in_channels != out_channels:
            self.shortcut =  nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 1, padding=0),
                    nn.BatchNorm2d(out_channels)
                )

    def forward(self, x):
        # shortcut(x) plays the role of "x", do as little work as possible to keep the tensor shapes the same.
        return self.shortcut(x) + self.F(x)

In [ ]:
cnn_res_model = nn.Sequential(
    ResidualBottleNeck(C, n_filters), #BottleNeck to start because we need more channels. Its also common to start with just one normal hidden layer before starting residual blocks.
    nn.LeakyReLU(leak_rate), #We are inserting a activation after each residual. This is optional.
    ResidualBlockE(n_filters),
    nn.LeakyReLU(leak_rate),
    nn.MaxPool2d((2,2)),
    ResidualBottleNeck(n_filters, 2*n_filters),
    nn.LeakyReLU(leak_rate),
    ResidualBlockE(2*n_filters),
    nn.LeakyReLU(leak_rate),
    nn.MaxPool2d((2,2)),
    ResidualBottleNeck(2*n_filters, 4*n_filters),
    nn.LeakyReLU(leak_rate),
    ResidualBlockE(4*n_filters),
    nn.LeakyReLU(leak_rate),
    nn.Flatten(),
    nn.Linear(D*n_filters//4, classes),
)

In [ ]:
cnn_res_results = train_network(cnn_res_model, loss_func, train_loader, test_loader=test_loader, epochs=10, score_funcs={'Accuracy': accuracy_score}, device=device)

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_results, label='CNN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_relu_results, label='CNN-ReLU')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_bn_results, label='CNN-ReLU-BN')
sns.lineplot(x='epoch', y='test Accuracy', data=cnn_res_results, label='CNN-ReLU-BN-Res')

#OPTIONAL

In [ ]:
zip_file_url = "https://download.pytorch.org/tutorial/data.zip"

import requests, zipfile, io
r = requests.get(zip_file_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

#Zip file is organized as data/names/[LANG].txt , where [LANG] is a specific language

namge_language_data = {}

#We will use some code to remove UNICODE tokens to make life easy for us processing wise
#e.g., convert something like "Ślusàrski" to Slusarski
import unicodedata
import string

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)
alphabet = {}
for i in range(n_letters):
    alphabet[all_letters[i]] = i

# Turn a Unicode string to plain ASCII, thanks to https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )


#Loop through every language, open the zip file entry, and read all the lines from the text file.
for zip_path in z.namelist():
    if "data/names/" in zip_path and zip_path.endswith(".txt"):
        lang = zip_path[len("data/names/"):-len(".txt")]
        with z.open(zip_path) as myfile:
            lang_names = [unicodeToAscii(line).lower() for line in str(myfile.read(), encoding='utf-8').strip().split("\n")]
            namge_language_data[lang] = lang_names

In [ ]:
dataset = LanguageNameDataset(namge_language_data, alphabet)#Reusing our code from chapter 4

train_lang_data, test_lang_data = torch.utils.data.random_split(dataset, (len(dataset)-300, 300))
train_lang_loader = DataLoader(train_lang_data, batch_size=32, shuffle=True, collate_fn=pad_and_pack)
test_lang_loader = DataLoader(test_lang_data, batch_size=32, shuffle=False, collate_fn=pad_and_pack)

In [ ]:
set_seed(42)

In [ ]:
rnn_3layer = nn.Sequential( #Simple old style RNN
  EmbeddingPackable(nn.Embedding(len(all_letters), 64)), #(B, T) -> (B, T, D)
  nn.RNN(64, n, num_layers=3, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )
  LastTimeStep(rnn_layers=3), #We need to take the RNN output and reduce it to one item, (B, D)
  nn.Linear(n, len(namge_language_data)), #(B, D) -> (B, classes)
)

#Apply gradient cliping to maximize its performance
for p in rnn_3layer.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -5, 5))

rnn_results = train_network(rnn_3layer, loss_func, train_lang_loader, test_loader=test_lang_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=10)

In [ ]:
lstm_3layer = nn.Sequential(
  EmbeddingPackable(nn.Embedding(len(all_letters), 64)), #(B, T) -> (B, T, D)
  #nn.RNN became nn.LSTM, and now we are upgraded to LSTMs w/ peephole connections
  nn.LSTM(64, n, num_layers=3, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )
  LastTimeStep(rnn_layers=3), #We need to take the RNN output and reduce it to one item, (B, D)
  nn.Linear(n, len(namge_language_data)), #(B, D) -> (B, classes)
)
#We still want to use gradient clipping with every kind of RNN, including LSTMs
for p in lstm_3layer.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -5, 5))

lstm_results = train_network(lstm_3layer, loss_func, train_lang_loader, test_loader=test_lang_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=10)

In [ ]:
sns.lineplot(x='epoch', y='test Accuracy', data=rnn_results, label='RNN: 3-Layer BiDir')
sns.lineplot(x='epoch', y='test Accuracy', data=lstm_results, label='LSTM: 3-Layer BiDir')